# Telegram Chatbot —

**Project structure preserved from original repo:**
```
telegram_chatbot/
├── app.py                          ← Telegram bot entry‑point (aiogram)
├── all-utils/
│   ├── main.py                     ← Orchestrates all utilities
│   └── utilities/
│       ├── pydantic_models.py      ← Request / Response validation
│       ├── query_validation_transformation.py
│       ├── logging_example.py
│       └── mem0_example.py         ← Long‑term memory (Mem0 + ChromaDB)
```

**LLM stack:** Groq (`llama-3.1-8b-instant` for speed / `llama-3.3-70b-versatile` for quality)  
**Memory:** Mem0 backed by ChromaDB (free, local)  
**Web search:** Tavily (free tier)  
**Agent framework:** LangChain ≥ 0.2  

---
**Phases**
1. Install dependencies
2. Write `.py` source files
3. Demo — Pydantic models
4. Demo — Query validation & transformation
5. Demo — Logging utility
6. Demo — Mem0 long‑term memory
7. LangChain agent (Groq + Tavily)
8. Collect outputs & zip everything


## Phase 1 — Install dependencies

In [1]:
# Install all required packages.
# Using LangChain >=0.2 ecosystem, Groq, Mem0, ChromaDB, Tavily, aiogram, pydantic v2.
!pip install -q \
    langchain\
    langchain-groq \
    langchain-community\
    langchain-core \
    langgraph\
    tavily-python\
    mem0ai \
    chromadb\
    pydantic[email]\
    python-dotenv \
    aiogram \
    groq


In [2]:
# ─── Install/Upgrade LangChain packages to ensure module visibility ───────────
!pip install --upgrade --quiet langchain langchain-community langchain-tavily

In [3]:
# Install sentence-transformers for local free embeddings used by Mem0
!pip install -q sentence-transformers


## Phase 2 — Write project source files

Each cell below writes a `.py` file to disk, preserving the original repo structure.

In [1]:
import os

# Create directory structure mirroring the original repo
os.makedirs('telegram_chatbot/all-utils/utilities', exist_ok=True)
os.makedirs('telegram_chatbot/all-utils/db', exist_ok=True)
os.makedirs('telegram_chatbot/research', exist_ok=True)
print('Directory structure created.')


Directory structure created.


### 2a — `utilities/pydantic_models.py`

**Purpose:** Defines strongly‑typed Pydantic v2 request/response schemas used throughout the pipeline. In GenAI apps these schemas enforce LLM structured output and guard against hallucinated field values.

In [2]:
# Write pydantic_models.py
PYDANTIC_SRC = '''"""
pydantic_models.py
==================
Pydantic v2 data models for the Telegram Chatbot pipeline.

In a GenAI context:
  - SearchRequest  → validated input arriving from the Telegram bot / LLM tool‑call.
  - SearchResponse → structured output returned to the caller / downstream chain.
"""

from pydantic import BaseModel, EmailStr, Field, field_validator
from datetime import datetime
from typing import Optional, List


# ── Input Schema ──────────────────────────────────────────────────────────────
class SearchRequest(BaseModel):
    """Represents a validated search request from the user."""

    user_id: str = Field(..., min_length=3, max_length=50, description="Unique user identifier")
    email: EmailStr = Field(..., description="Verified e‑mail of the requester")
    query: str = Field(..., min_length=1, max_length=200, description="Search query text")
    tags: Optional[List[str]] = Field(default_factory=list, description="Optional categorisation tags")

    @field_validator("query")
    @classmethod
    def query_must_not_be_empty(cls, value: str) -> str:
        """Reject queries that are only whitespace."""
        if not value.strip():
            raise ValueError("Query must not be empty or whitespace")
        return value.strip()


# ── Output Schema ─────────────────────────────────────────────────────────────
class SearchResponse(BaseModel):
    """Represents the validated response returned from the search handler."""

    status: str = Field(..., description="\'success\' or \'error\'")
    message: str = Field(..., description="Human‑readable status message")
    result_count: int = Field(0, ge=0, description="Number of results returned")
    results: List[dict] = Field(default_factory=list, description="List of result payloads")
    processed_at: datetime = Field(default_factory=datetime.utcnow, description="UTC timestamp")


# ── Factory helper ────────────────────────────────────────────────────────────
def build_search_response(request: SearchRequest) -> SearchResponse:
    """
    Simulate a search and wrap results in a validated SearchResponse.

    In a real pipeline this would call a vector DB or external search API.
    """
    example_results = [
        {"id": 1, "title": "Example item", "query": request.query},
    ]
    return SearchResponse(
        status="success",
        message=f"Search completed for user {request.user_id}",
        result_count=len(example_results),
        results=example_results,
    )


# ── Quick smoke‑test ──────────────────────────────────────────────────────────
def demo() -> None:
    """Run a quick validation demo."""
    payload = {
        "user_id": "user123",
        "email": "user@example.com",
        "query": "search for utilities",
        "tags": ["example", "demo"],
    }
    req = SearchRequest(**payload)
    resp = build_search_response(req)
    print("Request model:")
    print(req.model_dump_json(indent=2))
    print("\nResponse model:")
    print(resp.model_dump_json(indent=2))


if __name__ == "__main__":
    demo()
'''

with open('telegram_chatbot/all-utils/utilities/pydantic_models.py', 'w') as f:
    f.write(PYDANTIC_SRC)
print('pydantic_models.py written.')


pydantic_models.py written.


### 2b — `utilities/query_validation_transformation.py`

**Purpose:** Normalises, validates and transforms raw user input before it reaches the LLM or vector‑store retriever. Reduces token cost and retrieval noise.

In [3]:
QUERY_SRC = '''"""
query_validation_transformation.py
====================================
Validates and transforms raw user queries before they are sent to an LLM
or a vector‑store retriever.

GenAI rationale
---------------
* Removes stop‑words  → shorter prompts → lower token cost.
* Synonym mapping     → stable cache keys across semantically‑identical queries.
* Pattern guard       → rejects prompt‑injection attempts and garbage input.
* Signature field     → used as a deterministic cache key to skip redundant LLM calls.
"""

import re
from typing import Dict


# ── Constants ─────────────────────────────────────────────────────────────────

# Only allow safe printable characters (block prompt injection via weird Unicode)
ALLOWED_QUERY_PATTERN = re.compile(r"^[a-zA-Z0-9\s?@#\-_.,\'"()]+$")

# High‑frequency words that add noise to retrieval but carry little semantics
STOP_WORDS = {"the", "is", "and", "or", "for", "a", "an", "to"}

# Map colloquial terms to canonical forms for consistent cache keys
SYNONYMS = {
    "buy": "purchase",
    "find": "search",
    "latest": "recent",
}


# ── Validation ────────────────────────────────────────────────────────────────

def validate_query(query: str) -> bool:
    """
    Raise ValueError if the query is too short or contains unsafe characters.

    Parameters
    ----------
    query : str  Raw query string from the user.

    Returns
    -------
    bool  True if validation passes.
    """
    if not query or len(query.strip()) < 3:
        raise ValueError("Query must be at least 3 characters long.")
    if not ALLOWED_QUERY_PATTERN.match(query):
        raise ValueError("Query contains invalid characters.")
    return True


# ── Transformation ────────────────────────────────────────────────────────────

def transform_query(query: str) -> Dict[str, str]:
    """
    Normalise, stop‑word filter, and synonym‑map a raw query.

    Returns a dict with four keys:
        original   – the unmodified input
        normalized – lower‑cased, de‑duped whitespace
        cleaned    – stop‑words removed, synonyms applied
        signature  – underscore‑delimited form used as cache key
    """
    normalized = query.strip().lower()
    normalized = re.sub(r"\s+", " ", normalized)

    tokens = normalized.split()
    tokens = [SYNONYMS.get(tok, tok) for tok in tokens if tok not in STOP_WORDS]
    cleaned = " ".join(tokens)

    return {
        "original": query,
        "normalized": normalized,
        "cleaned": cleaned,
        "signature": cleaned.replace(" ", "_"),
    }


# ── Public entry‑point ────────────────────────────────────────────────────────

def handle_query(query: str) -> Dict[str, str]:
    """Validate then transform a query.  Raises ValueError on bad input."""
    validate_query(query)
    return transform_query(query)
'''

with open('telegram_chatbot/all-utils/utilities/query_validation_transformation.py', 'w') as f:
    f.write(QUERY_SRC)
print('query_validation_transformation.py written.')


query_validation_transformation.py written.


<>:22: SyntaxWarning: invalid escape sequence '\s'
<>:22: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_12671/3824924317.py:22: SyntaxWarning: invalid escape sequence '\s'
  ALLOWED_QUERY_PATTERN = re.compile(r"^[a-zA-Z0-9\s?@#\-_.,\'"()]+$")


### 2c — `utilities/logging_example.py`

**Purpose:** Dual‑sink logger (console + rotating file). In GenAI pipelines, structured logs capture every LLM prompt/response pair and token‑usage figures for cost auditing and debugging hallucinations.

In [4]:
LOGGING_SRC = '''"""
logging_example.py
==================
Provides a dual‑sink application logger (stdout + file) for the pipeline.

GenAI rationale
---------------
LLM calls are non‑deterministic.  Without structured logs you cannot:
  * Audit what prompt caused a hallucination.
  * Measure p99 latency of LLM API calls.
  * Track token usage for cost management.
  * Catch API rate‑limit / context‑overflow errors.
"""

import logging
import sys


# ── Logger factory ────────────────────────────────────────────────────────────

def get_app_logger(name: str = __name__) -> logging.Logger:
    """
    Return a named logger with a console handler (INFO+) and a file handler (DEBUG+).

    Idempotent: calling twice with the same name returns the cached logger.

    Parameters
    ----------
    name : str  Logger name, typically the module name.
    """
    logger = logging.getLogger(name)
    if logger.handlers:          # already configured — avoid duplicate handlers
        return logger

    logger.setLevel(logging.DEBUG)

    # Console — INFO and above for human readability during development
    console = logging.StreamHandler(sys.stdout)
    console.setLevel(logging.INFO)
    console.setFormatter(logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    ))

    # File — DEBUG and above for full audit trail
    file_h = logging.FileHandler(
        "telegram_chatbot/all-utils/utility_logging_example.log", encoding="utf-8"
    )
    file_h.setLevel(logging.DEBUG)
    file_h.setFormatter(logging.Formatter(
        "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
    ))

    logger.addHandler(console)
    logger.addHandler(file_h)
    logger.propagate = False     # prevent duplicate messages from root logger
    return logger


# ── Demo function ─────────────────────────────────────────────────────────────

def run_logging_demo() -> None:
    """Exercise every log level to verify the logger is wired correctly."""
    logger = get_app_logger("utility_logger")

    logger.debug("Debugging values: %s", {"step": 1, "status": "starting"})
    logger.info("Application example started.")
    logger.warning("This is a warning example for the logging utility.")

    try:
        _ = 10 / 0
    except ZeroDivisionError:
        logger.exception("An exception occurred while dividing by zero.")

    logger.info("Logging demo finished.")


if __name__ == "__main__":
    run_logging_demo()
'''

with open('telegram_chatbot/all-utils/utilities/logging_example.py', 'w') as f:
    f.write(LOGGING_SRC)
print('logging_example.py written.')


logging_example.py written.


### 2d — `utilities/mem0_example.py`

**Purpose:** Long‑term user‑memory store using Mem0 + ChromaDB (local, free).  Replaces the original OpenAI LLM back‑end with Groq so no paid OpenAI key is needed.

In [5]:
MEM0_SRC = '''"""
mem0_example.py
===============
Demonstrates long‑term memory management with Mem0 backed by ChromaDB
and the Groq LLM (llama‑3.1‑8b‑instant).

GenAI rationale
---------------
LLMs are stateless.  Mem0 adds a persistent, semantically‑searchable
memory layer so the chatbot can:
  * Remember user preferences across sessions.
  * Detect and resolve conflicting user facts (e.g. AWS → GCP migration).
  * Inject only the *relevant* memories into each prompt (RAG for preferences).
  * Provide an audit trail of how a user\'s profile evolved over time.
"""

import os
from mem0 import Memory


def run_observability_demo() -> None:
    """
    Full lifecycle: store → update → inspect history → semantic search.

    Requires GROQ_API_KEY in the environment (set via Colab secrets or os.environ).
    Uses ChromaDB as the local free vector store (no external service needed).
    """

    # ── Configuration: Groq LLM + local ChromaDB vector store ────────────────
    config = {
        "vector_store": {
            "provider": "chroma",
            "config": {
                "collection_name": "telegram_bot_demo",
                "path": "telegram_chatbot/all-utils/db",  # persisted locally
            },
        },
        "llm": {
            "provider": "groq",
            "config": {
                # llama‑3.1‑8b‑instant: fast, 8 k context — good for memory ops
                "model": "llama-3.1-8b-instant",
                "temperature": 0,
                "api_key": os.environ.get("GROQ_API_KEY"),
            },
        },
        # Embeddings: use a free local sentence‑transformers model via HuggingFace
        "embedder": {
            "provider": "huggingface",
            "config": {"model": "multi-qa-MiniLM-L6-cos-v1"},
        },
    }

    # ── Initialise the memory object ──────────────────────────────────────────
    print("\n[Mem0] Initialising memory store …")
    m = Memory.from_config(config)
    user_id = "colab_demo_user"

    # ── Step 1: Store an initial preference ──────────────────────────────────
    print("\n--- [Step 1] Storing initial preference ---")
    result = m.add("I prefer using FastAPI and AWS for my projects.", user_id=user_id)
    print(f"Result: {result}")

    # Robust ID extraction (Mem0 API returns either list or dict depending on version)
    mem_id = None
    if isinstance(result, list) and result:
        mem_id = result[0].get("id")
    elif isinstance(result, dict):
        res_list = result.get("results") or result.get("memories") or []
        if res_list:
            mem_id = res_list[0].get("id")

    # ── Step 2: Update the preference (triggers LLM‑driven conflict resolution)
    print("\n--- [Step 2] Updating preference (triggers change detection) ---")
    m.add(
        "Actually, I have decided to migrate all my projects to Google Cloud.",
        user_id=user_id,
    )

    # ── Step 3: Observability — inspect the evolution history ─────────────────
    print("\n--- [Step 3] OBSERVABILITY — Memory evolution report ---")
    if mem_id:
        history = m.history(memory_id=mem_id)
        for entry in history:
            event = entry.get("event", "unknown")
            old = entry.get("old_memory") or entry.get("old_value") or "Initial"
            new = entry.get("memory") or entry.get("new_value", "")
            print(f"  Event : {event}")
            print(f"  Old   : {old}")
            print(f"  New   : {new}")
            print("  " + "-" * 40)
    else:
        print("  Note: memory ID not captured — check the db folder for persisted data.")

    # ── Step 4: Semantic search using the updated profile ─────────────────────
    print("\n--- [Step 4] Semantic search: \'What is my deployment preference?\' ---")
    search_results = m.search(
        "What is my deployment preference?",
        filters={"user_id": user_id},
    )

    # Normalise search result format across Mem0 versions
    memories = (
        search_results.get("results")
        if isinstance(search_results, dict)
        else search_results
    )
    if memories:
        for res in memories:
            val = res.get("memory") or res.get("payload", {}).get("value")
            print(f"  Observed Memory : {val}  (Score: {res.get(\'score\')})")
    else:
        print("  No memories found for this user.")


if __name__ == "__main__":
    run_observability_demo()
'''

with open('telegram_chatbot/all-utils/utilities/mem0_example.py', 'w') as f:
    f.write(MEM0_SRC)
print('mem0_example.py written.')


mem0_example.py written.


### 2e — `all-utils/main.py`

Orchestration entry‑point — imports and runs every utility in sequence.

In [6]:
ALLUTILS_MAIN = '''"""
main.py  (all-utils)
=====================
Orchestrates all utility demos in sequence:
  1. Pydantic request/response models
  2. Query validation & transformation
  3. Logging
  4. Mem0 long‑term memory
"""

from utilities.pydantic_models import demo as pydantic_demo
from utilities.query_validation_transformation import handle_query
from utilities.logging_example import run_logging_demo
from utilities.mem0_example import run_observability_demo


def main() -> None:
    print("=" * 60)
    print("PHASE A — Pydantic Models")
    print("=" * 60)
    pydantic_demo()

    print("\n" + "=" * 60)
    print("PHASE B — Query Validation & Transformation")
    print("=" * 60)
    result = handle_query("Find the latest laptop deals")
    for k, v in result.items():
        print(f"  {k:12s}: {v}")

    print("\n" + "=" * 60)
    print("PHASE C — Logging")
    print("=" * 60)
    run_logging_demo()

    print("\n" + "=" * 60)
    print("PHASE D — Mem0 Observability")
    print("=" * 60)
    run_observability_demo()


if __name__ == "__main__":
    main()
'''

with open('telegram_chatbot/all-utils/main.py', 'w') as f:
    f.write(ALLUTILS_MAIN)
print('all-utils/main.py written.')


all-utils/main.py written.


### 2f — `app.py` (Telegram Bot entry‑point)

Converts the original aiogram + OpenAI bot to use **Groq** (`llama-3.3-70b-versatile`) for chat completions.  Docker / server code excluded as requested.

In [7]:
APP_SRC = '''"""
app.py
======
Telegram Chatbot entry‑point using aiogram 2.x and Groq LLM.

Original bot used openai.ChatCompletion (deprecated).
This version replaces it with the Groq client using:
  - llama-3.1-8b-instant  : fast responses, 8 k context.
  - llama-3.3-70b-versatile: richer answers when needed (swap model_name).

Token‑limit awareness
---------------------
* The Groq context window for llama‑3.1‑8b‑instant is 8 192 tokens.
* We store only the *last assistant response* in `reference.response` to
  avoid blowing the context on long conversations.
* For production use the Mem0 utility provides a better memory strategy.

Run locally:
    export GROQ_API_KEY=...
    export TELEGRAM_BOT_TOKEN=...
    python app.py
"""

import os
import logging
from aiogram import Bot, Dispatcher, executor, types
from groq import Groq

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger(__name__)

# ── API clients ───────────────────────────────────────────────────────────────
GROQ_API_KEY     = os.getenv("GROQ_API_KEY")
TELEGRAM_TOKEN   = os.getenv("TELEGRAM_BOT_TOKEN")

groq_client = Groq(api_key=GROQ_API_KEY)

# Model choices:
#   "llama-3.1-8b-instant"    – 8 k ctx, fastest
#   "llama-3.3-70b-versatile" – 32 k ctx, highest quality
MODEL_NAME = "llama-3.1-8b-instant"

# ── Reference memory ──────────────────────────────────────────────────────────

class Reference:
    """Stores the most recent assistant reply for single‑turn context."""
    def __init__(self) -> None:
        self.response: str = ""


reference = Reference()
bot        = Bot(token=TELEGRAM_TOKEN)
dispatcher = Dispatcher(bot)


# ── Helpers ───────────────────────────────────────────────────────────────────

def clear_past() -> None:
    """Reset the rolling conversation reference."""
    reference.response = ""


# ── Handlers ──────────────────────────────────────────────────────────────────

@dispatcher.message_handler(commands=["clear"])
async def clear(message: types.Message) -> None:
    """Clear the conversation context."""
    clear_past()
    await message.reply("Context cleared.  Starting a fresh conversation.")


@dispatcher.message_handler(commands=["start"])
async def welcome(message: types.Message) -> None:
    """/start — greet the user."""
    await message.reply(
        "Hi!  I am your AI assistant powered by Groq + LLaMA.\n"
        "Send me any message to chat.  Use /clear to reset context, /help for more info."
    )


@dispatcher.message_handler(commands=["help"])
async def helper(message: types.Message) -> None:
    """/help — show available commands."""
    await message.reply(
        "Available commands:\n"
        "  /start — Start or restart the conversation.\n"
        "  /clear — Wipe conversation history.\n"
        "  /help  — Show this menu.\n\n"
        "Just type a message to chat with the LLaMA model via Groq!"
    )


@dispatcher.message_handler()
async def chat_handler(message: types.Message) -> None:
    """
    Handle any free‑text message.

    Sends the user message plus the last assistant reply (rolling 1‑turn memory)
    to the Groq API and echoes the response back to Telegram.

    Token‑limit guard: if the previous response is very long we truncate it to
    keep the total prompt under ~6 000 tokens (conservative for 8 k window).
    """
    user_text = message.text
    logger.info("User (%s): %s", message.from_user.username, user_text)

    # Truncate previous response if it exceeds ~1 500 characters to stay safe
    prev_context = reference.response[:1500] if reference.response else ""

    messages = []
    if prev_context:
        messages.append({"role": "assistant", "content": prev_context})
    messages.append({"role": "user", "content": user_text})

    try:
        completion = groq_client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            max_tokens=1024,      # cap reply length to control costs
            temperature=0.7,
        )
        reply = completion.choices[0].message.content
    except Exception as exc:
        logger.exception("Groq API error: %s", exc)
        reply = "Sorry, I encountered an error.  Please try again."

    reference.response = reply
    logger.info("Bot: %s", reply[:120])
    await bot.send_message(chat_id=message.chat.id, text=reply)


# ── Entry‑point ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    logger.info("Starting Telegram bot with model: %s", MODEL_NAME)
    executor.start_polling(dispatcher, skip_updates=False)
'''

with open('telegram_chatbot/app.py', 'w') as f:
    f.write(APP_SRC)
print('app.py written.')


app.py written.


In [8]:
# Write requirements.txt at project root
reqs = '''aiogram==2.25.1
groq
langchain>=0.2.0
langchain-groq>=0.1.0
langchain-community>=0.2.0
langchain-core>=0.2.0
langgraph>=0.1.0
tavily-python>=0.3.0
mem0ai>=0.1.0
chromadb>=0.5.0
pydantic[email]>=2.0,<3.0
python-dotenv
sentence-transformers
'''
with open('telegram_chatbot/requirements.txt', 'w') as f:
    f.write(reqs)
print('requirements.txt written.')


requirements.txt written.


## Phase 3 — Demo: Pydantic Request & Response Models

Validated models ensure the chatbot never passes malformed data into the LLM chain.

In [9]:
import os

# Re-write pydantic_models.py to ensure it's not corrupted
PYDANTIC_SRC = '''"""
pydantic_models.py
==================
Pydantic v2 data models for the Telegram Chatbot pipeline.

In a GenAI context:
  - SearchRequest  → validated input arriving from the Telegram bot / LLM tool‑call.
  - SearchResponse → structured output returned to the caller / downstream chain.
"""

from pydantic import BaseModel, EmailStr, Field, field_validator
from datetime import datetime
from typing import Optional, List


# ── Input Schema ──────────────────────────────────────────────────────────────
class SearchRequest(BaseModel):
    """Represents a validated search request from the user."""

    user_id: str = Field(..., min_length=3, max_length=50, description="Unique user identifier")
    email: EmailStr = Field(..., description="Verified e‑mail of the requester")
    query: str = Field(..., min_length=1, max_length=200, description="Search query text")
    tags: Optional[List[str]] = Field(default_factory=list, description="Optional categorisation tags")

    @field_validator("query")
    @classmethod
    def query_must_not_be_empty(cls, value: str) -> str:
        """Reject queries that are only whitespace."""
        if not value.strip():
            raise ValueError("Query must not be empty or whitespace")
        return value.strip()


# ── Output Schema ─────────────────────────────────────────────────────────────
class SearchResponse(BaseModel):
    """Represents the validated response returned from the search handler."""

    status: str = Field(..., description="'success' or 'error'")
    message: str = Field(..., description="Human‑readable status message")
    result_count: int = Field(0, ge=0, description="Number of results returned")
    results: List[dict] = Field(default_factory=list, description="List of result payloads")
    processed_at: datetime = Field(default_factory=datetime.utcnow, description="UTC timestamp")


# ── Factory helper ────────────────────────────────────────────────────────────
def build_search_response(request: SearchRequest) -> SearchResponse:
    """
    Simulate a search and wrap results in a validated SearchResponse.

    In a real pipeline this would call a vector DB or external search API.
    """
    example_results = [
        {"id": 1, "title": "Example item", "query": request.query},
    ]
    return SearchResponse(
        status="success",
        message=f"Search completed for user {request.user_id}",
        result_count=len(example_results),
        results=example_results,
    )


# ── Quick smoke‑test ──────────────────────────────────────────────────────────
def demo() -> None:
    """Run a quick validation demo."""
    payload = {
        "user_id": "user123",
        "email": "user@example.com",
        "query": "search for utilities",
        "tags": ["example", "demo"],
    }
    req = SearchRequest(**payload)
    resp = build_search_response(req)
    print("Request model:")
    print(req.model_dump_json(indent=2))
    print()
    print("Response model:")
    print(resp.model_dump_json(indent=2))


if __name__ == "__main__":
    demo()
'''
with open('telegram_chatbot/all-utils/utilities/pydantic_models.py', 'w') as f:
    f.write(PYDANTIC_SRC)

import sys
sys.path.insert(0, 'telegram_chatbot/all-utils')

from utilities.pydantic_models import SearchRequest, build_search_response

# ── Valid request ─────────────────────────────────────────────────────────────
print('=== Valid request ===')
req = SearchRequest(
    user_id='user123',
    email='user@example.com',
    query='Find the latest news on LangChain',
    tags=['ai', 'langchain'],
)
resp = build_search_response(req)
print(req.model_dump_json(indent=2))
print(resp.model_dump_json(indent=2))

# ── Invalid request (should raise) ───────────────────────────────────────────
print('\n=== Invalid request — blank query ===')
from pydantic import ValidationError
try:
    bad = SearchRequest(user_id='u1', email='bad-email', query='   ')
except ValidationError as e:
    print('ValidationError caught (expected):')
    for err in e.errors():
        print(f'  field={err["loc"]}  msg={err["msg"]}')

=== Valid request ===
{
  "user_id": "user123",
  "email": "user@example.com",
  "query": "Find the latest news on LangChain",
  "tags": [
    "ai",
    "langchain"
  ]
}
{
  "status": "success",
  "message": "Search completed for user user123",
  "result_count": 1,
  "results": [
    {
      "id": 1,
      "title": "Example item",
      "query": "Find the latest news on LangChain"
    }
  ],
  "processed_at": "2026-05-19T04:28:57.643827"
}

=== Invalid request — blank query ===
ValidationError caught (expected):
  field=('user_id',)  msg=String should have at least 3 characters
  field=('email',)  msg=value is not a valid email address: An email address must have an @-sign.
  field=('query',)  msg=Value error, Query must not be empty or whitespace


## Phase 4 — Demo: Query Validation & Transformation

In [10]:
import os
import sys

# Corrected QUERY_SRC content
QUERY_SRC_FIXED = '''"""
query_validation_transformation.py
====================================
Validates and transforms raw user queries before they are sent to an LLM
or a vector‑store retriever.

GenAI rationale
---------------
* Removes stop‑words  → shorter prompts → lower token cost.
* Synonym mapping     → stable cache keys across semantically‑identical queries.
* Pattern guard       → rejects prompt‑injection attempts and garbage input.
* Signature field     → used as a deterministic cache key to skip redundant LLM calls.
"""

import re
from typing import Dict


# ── Constants ─────────────────────────────────────────────────────────────────

# Only allow safe printable characters (block prompt injection via weird Unicode)
ALLOWED_QUERY_PATTERN = re.compile(r"^[a-zA-Z0-9\\s?@#\\-_.,'\\"()]+$")

# High‑frequency words that add noise to retrieval but carry little semantics
STOP_WORDS = {"the", "is", "and", "or", "for", "a", "an", "to"}

# Map colloquial terms to canonical forms for consistent cache keys
SYNONYMS = {
    "buy": "purchase",
    "find": "search",
    "latest": "recent",
}


# ── Validation ────────────────────────────────────────────────────────────────

def validate_query(query: str) -> bool:
    """
    Raise ValueError if the query is too short or contains unsafe characters.

    Parameters
    ----------
    query : str  Raw query string from the user.

    Returns
    -------
    bool  True if validation passes.
    """
    if not query or len(query.strip()) < 3:
        raise ValueError("Query must be at least 3 characters long.")
    if not ALLOWED_QUERY_PATTERN.match(query):
        raise ValueError("Query contains invalid characters.")
    return True


# ── Transformation ────────────────────────────────────────────────────────────

def transform_query(query: str) -> Dict[str, str]:
    """
    Normalise, stop‑word filter, and synonym‑map a raw query.

    Returns a dict with four keys:
        original   – the unmodified input
        normalized – lower‑cased, de‑duped whitespace
        cleaned    – stop‑words removed, synonyms applied
        signature  – underscore‑delimited form used as cache key
    """
    normalized = query.strip().lower()
    normalized = re.sub(r"\\s+", " ", normalized)

    tokens = normalized.split()
    tokens = [SYNONYMS.get(tok, tok) for tok in tokens if tok not in STOP_WORDS]
    cleaned = " ".join(tokens)

    return {
        "original": query,
        "normalized": normalized,
        "cleaned": cleaned,
        "signature": cleaned.replace(" ", "_"),
    }


# ── Public entry‑point ────────────────────────────────────────────────────────

def handle_query(query: str) -> Dict[str, str]:
    """Validate then transform a query.  Raises ValueError on bad input."""
    validate_query(query)
    return transform_query(query)
'''

# Rewrite the file with the corrected content
with open('telegram_chatbot/all-utils/utilities/query_validation_transformation.py', 'w') as f:
    f.write(QUERY_SRC_FIXED)

# Ensure the path is in sys.path
# This line is likely already present in previous cells, but good to ensure
sys.path.insert(0, 'telegram_chatbot/all-utils')

# Now import and run the original code
from utilities.query_validation_transformation import handle_query

test_queries = [
    'Find the latest laptop deals',
    'buy a new python book',
    'Where can I search for the latest AI news?',
]

for q in test_queries:
    result = handle_query(q)
    print(f'Original : {result["original"]}')
    print(f'Cleaned  : {result["cleaned"]}')
    print(f'Signature: {result["signature"]}')
    print('-' * 50)


Original : Find the latest laptop deals
Cleaned  : search recent laptop deals
Signature: search_recent_laptop_deals
--------------------------------------------------
Original : buy a new python book
Cleaned  : purchase new python book
Signature: purchase_new_python_book
--------------------------------------------------
Original : Where can I search for the latest AI news?
Cleaned  : where can i search recent ai news?
Signature: where_can_i_search_recent_ai_news?
--------------------------------------------------


## Phase 5 — Demo: Logging Utility

In [11]:
from utilities.logging_example import run_logging_demo

run_logging_demo()

# Verify the log file was created
import os
log_path = 'telegram_chatbot/all-utils/utility_logging_example.log'
if os.path.exists(log_path):
    print(f'\nLog file created at {log_path}')
    with open(log_path) as f:
        print(f.read())


2026-05-19 04:29:04,616 - utility_logger - INFO - Application example started.
2026-05-19 04:29:04,617 - utility_logger - WARNING - This is a warning example for the logging utility.
2026-05-19 04:29:04,619 - utility_logger - ERROR - An exception occurred while dividing by zero.
Traceback (most recent call last):
  File "/content/telegram_chatbot/all-utils/utilities/logging_example.py", line 70, in run_logging_demo
    _ = 10 / 0
        ~~~^~~
ZeroDivisionError: division by zero
2026-05-19 04:29:04,621 - utility_logger - INFO - Logging demo finished.

Log file created at telegram_chatbot/all-utils/utility_logging_example.log
2026-05-19 04:16:00,319 | DEBUG | utility_logger | Debugging values: {'step': 1, 'status': 'starting'}
2026-05-19 04:16:00,319 | INFO | utility_logger | Application example started.
2026-05-19 04:16:00,321 | WARNING | utility_logger | This is a warning example for the logging utility.
2026-05-19 04:16:00,322 | ERROR | utility_logger | An exception occurred while d

## Phase 6 — Demo: Mem0 Long‑term Memory

Uses **ChromaDB** (local, free) as the vector store and **Groq** as the LLM for memory operations.  No OpenAI key required.

In [12]:
# Load GROQ_API_KEY from Colab Secrets (set via the key icon in the left sidebar)
from google.colab import userdata
import os

os.environ['GROQ_API_KEY']   = userdata.get('GROQ_API_KEY')
os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')

print('API keys loaded from Colab Secrets.')


API keys loaded from Colab Secrets.


In [13]:
import os

# Corrected MEM0_SRC content to fix SyntaxError
MEM0_SRC_FIXED = '''"""
mem0_example.py
===============
Demonstrates long‑term memory management with Mem0 backed by ChromaDB
and the Groq LLM (llama‑3.1‑8b‑instant).

GenAI rationale
---------------
LLMs are stateless.  Mem0 adds a persistent, semantically‑searchable
memory layer so the chatbot can:
  * Remember user preferences across sessions.
  * Detect and resolve conflicting user facts (e.g. AWS → GCP migration).
  * Inject only the *relevant* memories into each prompt (RAG for preferences).
  * Provide an audit trail of how a user\\'s profile evolved over time.
"""

import os
from mem0 import Memory


def run_observability_demo() -> None:
    """
    Full lifecycle: store → update → inspect history → semantic search.

    Requires GROQ_API_KEY in the environment (set via Colab secrets or os.environ).
    Uses ChromaDB as the local free vector store (no external service needed).
    """

    # ── Configuration: Groq LLM + local ChromaDB vector store ────────────────
    config = {
        "vector_store": {
            "provider": "chroma",
            "config": {
                "collection_name": "telegram_bot_demo",
                "path": "telegram_chatbot/all-utils/db",  # persisted locally
            },
        },
        "llm": {
            "provider": "groq",
            "config": {
                # llama‑3.1‑8b‑instant: fast, 8 k context — good for memory ops
                "model": "llama-3.1-8b-instant",
                "temperature": 0,
                "api_key": os.environ.get("GROQ_API_KEY"),
            },
        },
        # Embeddings: use a free local sentence‑transformers model via HuggingFace
        "embedder": {
            "provider": "huggingface",
            "config": {"model": "multi-qa-MiniLM-L6-cos-v1"},
        },
    }

    # ── Initialise the memory object ──────────────────────────────────────────
    print()
    print("[Mem0] Initialising memory store …")
    m = Memory.from_config(config)
    user_id = "colab_demo_user"

    # ── Step 1: Store an initial preference ──────────────────────────────────
    print()
    print("--- [Step 1] Storing initial preference ---")
    result = m.add("I prefer using FastAPI and AWS for my projects.", user_id=user_id)
    print(f"Result: {result}")

    # Robust ID extraction (Mem0 API returns either list or dict depending on version)
    mem_id = None
    if isinstance(result, list) and result:
        mem_id = result[0].get("id")
    elif isinstance(result, dict):
        res_list = result.get("results") or result.get("memories") or []
        if res_list:
            mem_id = res_list[0].get("id")

    # ── Step 2: Update the preference (triggers LLM‑driven conflict resolution)
    print()
    print("--- [Step 2] Updating preference (triggers change detection) ---")
    m.add(
        "Actually, I have decided to migrate all my projects to Google Cloud.",
        user_id=user_id,
    )

    # ── Step 3: Observability — inspect the evolution history ─────────────────
    print()
    print("--- [Step 3] OBSERVABILITY — Memory evolution report ---")
    if mem_id:
        history = m.history(memory_id=mem_id)
        for entry in history:
            event = entry.get("event", "unknown")
            old = entry.get("old_memory") or entry.get("old_value") or "Initial"
            new = entry.get("memory") or entry.get("new_value", "")
            print(f"  Event : {event}")
            print(f"  Old   : {old}")
            print(f"  New   : {new}")
            print("  " + "-" * 40)
    else:
        print("  Note: memory ID not captured — check the db folder for persisted data.")

    # ── Step 4: Semantic search using the updated profile ─────────────────────
    print()
    print("--- [Step 4] Semantic search: \'What is my deployment preference?\' ---")
    search_results = m.search(
        "What is my deployment preference?",
        filters={"user_id": user_id},
    )

    # Normalise search result format across Mem0 versions
    memories = (
        search_results.get("results")
        if isinstance(search_results, dict)
        else search_results
    )
    if memories:
        for res in memories:
            val = res.get("memory") or res.get("payload", {}).get("value")
            print(f"  Observed Memory : {val}  (Score: {res.get(\'score\')})")
    else:
        print("  No memories found for this user.")


if __name__ == "__main__":
    run_observability_demo()
'''

# Rewrite the file with the corrected content
with open('telegram_chatbot/all-utils/utilities/mem0_example.py', 'w') as f:
    f.write(MEM0_SRC_FIXED)

# Ensure the path is in sys.path (already done in previous cells, but good for robustness)
import sys
if 'telegram_chatbot/all-utils' not in sys.path:
    sys.path.insert(0, 'telegram_chatbot/all-utils')

from utilities.mem0_example import run_observability_demo

run_observability_demo()


[Mem0] Initialising memory store …


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reus

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/mem0/embeddings/huggingface.py:27: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.config.embedding_dims = self.config.embedding_dims or self.model.get_sentence_embedding_dimension()
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.


--- [Step 1] Storing initial preference ---
Result: {'results': []}

--- [Step 2] Updating preference (triggers change detection) ---


ERROR:mem0.memory.main:LLM extraction failed: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01k8zbvz9xf3v97wzxqpfaxz6g` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 9786, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}



--- [Step 3] OBSERVABILITY — Memory evolution report ---
  Note: memory ID not captured — check the db folder for persisted data.

--- [Step 4] Semantic search: 'What is my deployment preference?' ---


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


  No memories found for this user.


## Phase 7 — LangChain Agent with Groq + Tavily Web Search

This phase wires together:
- **Groq** (`llama-3.3-70b-versatile`) as the reasoning LLM
- **Tavily** as the free web‑search tool
- **LangChain `create_react_agent`** (LCEL, LangChain ≥ 0.2) as the agent loop
- **ConversationBufferWindowMemory** for in‑session chat history

Token‑limit awareness: `llama-3.3-70b-versatile` has a 32 768‑token context window.  
We cap `max_tokens=2048` per completion and use a `k=5` window memory to stay well within limits.


In [15]:
# Check installed LangChain-related packages and versions

import sys
import pkg_resources

packages_to_check = [
    "langchain",
    "langchain-core",
    "langchain-community",
    "langchain-groq",
    "langchain-tavily",
    "langgraph",
    "tavily-python",
]

print(f"Python Version: {sys.version}\n")

for package in packages_to_check:
    try:
        version = pkg_resources.get_distribution(package).version
        print(f"{package:<25} INSTALLED  | Version: {version}")
    except pkg_resources.DistributionNotFound:
        print(f"{package:<25} NOT INSTALLED")

/tmp/ipykernel_12671/4224110118.py:4: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

langchain                 INSTALLED  | Version: 1.3.1
langchain-core            INSTALLED  | Version: 1.4.0
langchain-community       INSTALLED  | Version: 0.4.1
langchain-groq            INSTALLED  | Version: 1.1.2
langchain-tavily          INSTALLED  | Version: 0.2.18
langgraph                 INSTALLED  | Version: 1.2.0
tavily-python             INSTALLED  | Version: 0.7.24


In [16]:
# ─── Imports ───────────────────────────────────────────────────────────────
import os

from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch

from langchain.agents import create_agent

# ─── LLM ───────────────────────────────────────────────────────────────────
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.5,
    max_tokens=2048,
    api_key=os.environ["GROQ_API_KEY"],
)

# ─── Tavily Tool ───────────────────────────────────────────────────────────
search_tool = TavilySearch(
    max_results=3,
    tavily_api_key=os.environ["TAVILY_API_KEY"],
)

tools = [search_tool]

# ─── Create Agent ──────────────────────────────────────────────────────────
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful AI assistant.",
)

print("Agent initialized successfully.")

Agent initialized successfully.


In [17]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What are the latest AI developments today?"
            }
        ]
    }
)

print(response)

{'messages': [HumanMessage(content='What are the latest AI developments today?', additional_kwargs={}, response_metadata={}, id='4ae4a0a4-cd31-4bcb-82e2-5f97c1e144af'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '26x5z7x0m', 'function': {'arguments': '{"end_date":null,"exclude_domains":[],"include_domains":[],"include_images":false,"query":"latest AI developments","search_depth":"advanced","start_date":null,"time_range":"day","topic":"general"}', 'name': 'tavily_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 1697, 'total_tokens': 1763, 'completion_time': 0.170495668, 'completion_tokens_details': None, 'prompt_time': 0.173090761, 'prompt_tokens_details': None, 'queue_time': 0.22694094, 'total_time': 0.343586429}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [18]:
print(response["messages"][-1].content)

Based on the search results, the latest AI developments today include:

1. Large Language Models (LLMs) driving AI agent advancement, with developers creating agents with significantly enhanced natural language processing capabilities.
2. Domain specialization, with AI agents becoming the default solution for specific problem domains.
3. Productivity transformation, with AI agents evolving from simple chatbots to productivity powerhouses, enabling businesses to automate decisions and redefine workflows.
4. Enterprise adoption, with AI agents being adopted by enterprises to transform work processes at various levels, from programming to development.
5. API integration challenges, with experts noting that most organizations aren't agent-ready, and the real challenge being exposing enterprise APIs for agents to leverage effectively.
6. Foundation models for agency, with the development of foundation models specifically designed for agency, capable of reasoning through multi-step plans, us

In [19]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Latest AI developments today"
            }
        ]
    }
)

final_answer = response["messages"][-1].content

print(final_answer)

The latest AI developments include advancements in autonomous agents, robotics, healthcare AI, and global investments. Some of the key developments include the release of new AI models such as GPT-4.5 and the launch of premium AI agents. Additionally, there have been significant advancements in AI-powered tools for retail employees, customer service, inventory management, and DIY recommendations. The integration of AI and robotics is also transforming how work is done, enabling businesses to achieve new levels of productivity and innovation. Furthermore, the development of explainable AI (XAI) is driving the responsible adoption of AI by addressing ethical and societal concerns related to algorithmic transparency and bias. Overall, the latest AI developments are driving transformation across industries and creating massive opportunities for entrepreneurs willing to move fast.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [20]:
def ask_agent(query):
    response = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": query}
            ]
        }
    )

    return response["messages"][-1].content


print(ask_agent("What is LangGraph?"))

LangGraph is an open-source framework built by LangChain that streamlines the creation and management of AI agent workflows. It combines large language models (LLMs) with graph-based architectures, allowing developers to map, organize, and optimize how AI agents interact and make decisions. LangGraph offers a scalable, transparent, and developer-friendly way to design advanced AI systems, ranging from simple chatbots to multi-agent systems. It provides low-level supporting infrastructure for any long-running, stateful workflow or agent, including durable execution, streaming, human-in-the-loop, and more.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [21]:
print(response["messages"][-1].content)

The latest AI developments include advancements in autonomous agents, robotics, healthcare AI, and global investments. Some of the key developments include the release of new AI models such as GPT-4.5 and the launch of premium AI agents. Additionally, there have been significant advancements in AI-powered tools for retail employees, customer service, inventory management, and DIY recommendations. The integration of AI and robotics is also transforming how work is done, enabling businesses to achieve new levels of productivity and innovation. Furthermore, the development of explainable AI (XAI) is driving the responsible adoption of AI by addressing ethical and societal concerns related to algorithmic transparency and bias. Overall, the latest AI developments are driving transformation across industries and creating massive opportunities for entrepreneurs willing to move fast.


In [25]:
# ─────────────────────────────────────────────────────────────────────────────
# Install Required Packages (Run Once)
# ─────────────────────────────────────────────────────────────────────────────
# !pip install -U langchain langchain-core langchain-groq \
# langchain-tavily langgraph

# ─────────────────────────────────────────────────────────────────────────────
# Imports
# ─────────────────────────────────────────────────────────────────────────────
import os

from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain.agents import create_agent

# ─────────────────────────────────────────────────────────────────────────────
# LLM (Groq)
# ─────────────────────────────────────────────────────────────────────────────
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.5,
    max_tokens=2048,
    api_key=os.environ["GROQ_API_KEY"],
)

# ─────────────────────────────────────────────────────────────────────────────
# Tavily Web Search Tool
# ─────────────────────────────────────────────────────────────────────────────
search_tool = TavilySearch(
    max_results=3,
    tavily_api_key=os.environ["TAVILY_API_KEY"],
)

tools = [search_tool]

# ─────────────────────────────────────────────────────────────────────────────
# System Prompt
# ─────────────────────────────────────────────────────────────────────────────
system_prompt = """
You are a helpful AI assistant for a Telegram chatbot.

You have access to a web search tool.
Use it whenever current information is needed.

Always keep responses concise because users are on mobile devices.
"""

# ─────────────────────────────────────────────────────────────────────────────
# Create Modern LangChain v1 Agent
# ─────────────────────────────────────────────────────────────────────────────
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
)

print("Agent initialized successfully.")

# ─────────────────────────────────────────────────────────────────────────────
# Helper Function
# ─────────────────────────────────────────────────────────────────────────────
def ask_agent(query):
    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": query
                }
            ]
        }
    )

    return response["messages"][-1].content

# ─────────────────────────────────────────────────────────────────────────────
# Example Usage
# ─────────────────────────────────────────────────────────────────────────────
answer = ask_agent("Latest AI developments today")

print("\nAI Response:\n")
print(answer)

Agent initialized successfully.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



AI Response:

Here are the latest AI developments today:

1. GPT Image 2 + Seedance 2 is an insane combo
2. Flipbook reimagines the web as an infinite visual browser where every page is a visual representation of the content
3. Runway brought generative AI to the big screen with Everything Everywhere All At Once
4. Neeva became the first AI-native search engine
5. Federal AI spending has increased, with 28 agencies having AI contracts in 2026, up from 17 in 2022 and 23 in 2024.

These are just a few examples of the latest developments in AI. For more information, you can check out the links provided in the search results.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [27]:
# ─── Test the agent ───────────────────────────────────────────────────────────
# Save each run output for the final zip archive
phase7_outputs = []

test_questions = [
    'What is LangChain and why is it useful for building chatbots?',
    'Search the web: what are the latest Groq LLM models available in 2025?',
]

for q in test_questions:
    print(f'\n>>> Question: {q}')
    answer = ask_agent(q)
    phase7_outputs.append({'question': q, 'answer': answer})
    print(f'\n>>> Final Answer:\n{answer}')
    print('=' * 70)


>>> Question: What is LangChain and why is it useful for building chatbots?


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



>>> Final Answer:
LangChain is a powerful framework for building chatbots with features like memory, retrieval-augmented generation (RAG), and real-time search. It provides a simple way to integrate with Hugging Face's LLMs, enabling a chatbot to process a user's query and generate responses efficiently. LangChain can be used to build intelligent, memory-enhanced chatbots capable of leveraging text data for more accurate responses. It can also be used to create dynamic and scalable AI chat solutions.

>>> Question: Search the web: what are the latest Groq LLM models available in 2025?


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



>>> Final Answer:
The latest Groq LLM models available in 2025 are Grok-3 and Grok-4.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Phase 8 — Save All Outputs & Zip Archive

Collects every phase output into a single downloadable `.zip` file.

In [28]:
import json
import zipfile
import shutil
from datetime import datetime

OUTPUT_DIR = 'telegram_chatbot_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Phase 3 output: Pydantic demo ────────────────────────────────────────────
from utilities.pydantic_models import SearchRequest, build_search_response
req_out = SearchRequest(user_id='demo', email='demo@example.com', query='demo query')
resp_out = build_search_response(req_out)
with open(f'{OUTPUT_DIR}/phase3_pydantic_demo.json', 'w') as f:
    json.dump({'request': req_out.model_dump(mode='json'),
               'response': resp_out.model_dump(mode='json')}, f, indent=2, default=str)

# ── Phase 4 output: Query transformation ─────────────────────────────────────
from utilities.query_validation_transformation import handle_query
qv_results = [handle_query(q) for q in ['Find latest AI news', 'buy python book']]
with open(f'{OUTPUT_DIR}/phase4_query_transformation.json', 'w') as f:
    json.dump(qv_results, f, indent=2)

# ── Phase 5 output: Copy log file ─────────────────────────────────────────────
log_src = 'telegram_chatbot/all-utils/utility_logging_example.log'
if os.path.exists(log_src):
    shutil.copy(log_src, f'{OUTPUT_DIR}/phase5_logging.log')

# ── Phase 7 output: Agent Q&A ─────────────────────────────────────────────────
with open(f'{OUTPUT_DIR}/phase7_agent_qa.json', 'w') as f:
    json.dump(phase7_outputs, f, indent=2)

# ── Copy all .py source files ─────────────────────────────────────────────────
src_dir = f'{OUTPUT_DIR}/src'
shutil.copytree('telegram_chatbot', src_dir, dirs_exist_ok=True)

# ── Create the zip archive ────────────────────────────────────────────────────
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_name = f'telegram_chatbot_pipeline_{timestamp}.zip'

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        # Skip __pycache__ and .pyc files
        dirs[:] = [d for d in dirs if d != '__pycache__']
        for file in files:
            if not file.endswith('.pyc'):
                fp = os.path.join(root, file)
                zf.write(fp, os.path.relpath(fp, OUTPUT_DIR))

print(f'\nArchive created: {zip_name}')
print('\nContents:')
with zipfile.ZipFile(zip_name) as zf:
    for name in sorted(zf.namelist()):
        print(f'  {name}')



Archive created: telegram_chatbot_pipeline_20260519_043616.zip

Contents:
  phase3_pydantic_demo.json
  phase4_query_transformation.json
  phase5_logging.log
  phase7_agent_qa.json
  src/all-utils/db/chroma.sqlite3
  src/all-utils/main.py
  src/all-utils/utilities/logging_example.py
  src/all-utils/utilities/mem0_example.py
  src/all-utils/utilities/pydantic_models.py
  src/all-utils/utilities/query_validation_transformation.py
  src/all-utils/utility_logging_example.log
  src/app.py
  src/requirements.txt


/usr/local/lib/python3.12/dist-packages/pydantic/main.py:250: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [29]:
# Download the zip from Colab
from google.colab import files
files.download(zip_name)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>